# Oil Spill Detection Model (Step 1)

This notebook implements the first step of our two-stage framework:
**Binary classification to detect presence of oil spills in SAR imagery**

## Architecture:
- EfficientNet-B4 backbone with custom classifier
- Advanced training strategies (focal loss, mixup, etc.)
- Comprehensive evaluation metrics

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
import torchvision.models as models
from efficientnet_pytorch import EfficientNet

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import precision_recall_curve, average_precision_score
import pandas as pd
from tqdm import tqdm
import json
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Import preprocessing utilities
import sys
sys.path.append('.')
# from preprocessing import create_data_loaders  # Uncomment when preprocessing is available

plt.style.use('seaborn-v0_8')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Detection Model Architecture

In [ ]:
class OilSpillDetector(nn.Module):
    """Oil spill detection model using EfficientNet backbone"""
    
    def __init__(self, model_name='efficientnet-b4', num_classes=2, pretrained=True, dropout=0.3):
        super(OilSpillDetector, self).__init__()
        
        # Load EfficientNet backbone
        if pretrained:
            self.backbone = EfficientNet.from_pretrained(model_name, num_classes=num_classes)
        else:
            self.backbone = EfficientNet.from_name(model_name, num_classes=num_classes)
        
        # Modify first layer for single-channel input (SAR images)
        # Note: We'll use 3-channel input for compatibility with pretrained weights
        # The preprocessing will duplicate the SAR channel
        
        # Get the number of features from the backbone
        num_features = self.backbone._fc.in_features
        
        # Replace classifier with custom head
        self.backbone._fc = nn.Identity()  # Remove original classifier
        
        # Custom classification head
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Initialize classifier weights"""
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Extract features using backbone
        features = self.backbone(x)
        
        # Apply classifier
        output = self.classifier(features)
        
        return output
    
    def extract_features(self, x):
        """Extract features for analysis or transfer learning"""
        with torch.no_grad():
            features = self.backbone(x)
        return features

# Alternative: ResNet-based detector
class ResNetDetector(nn.Module):
    """Alternative detector using ResNet backbone"""
    
    def __init__(self, model_name='resnet50', num_classes=2, pretrained=True, dropout=0.3):
        super(ResNetDetector, self).__init__()
        
        # Load ResNet backbone
        if model_name == 'resnet50':
            self.backbone = models.resnet50(pretrained=pretrained)
        elif model_name == 'resnet101':
            self.backbone = models.resnet101(pretrained=pretrained)
        else:
            raise ValueError(f"Unsupported model: {model_name}")
        
        # Get number of features
        num_features = self.backbone.fc.in_features
        
        # Replace classifier
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

def create_model(config):
    """Create detection model based on configuration"""
    model_type = config.get('model_type', 'efficientnet')
    
    if model_type == 'efficientnet':
        model = OilSpillDetector(
            model_name=config.get('backbone', 'efficientnet-b4'),
            num_classes=config.get('num_classes', 2),
            pretrained=config.get('pretrained', True),
            dropout=config.get('dropout', 0.3)
        )
    elif model_type == 'resnet':
        model = ResNetDetector(
            model_name=config.get('backbone', 'resnet50'),
            num_classes=config.get('num_classes', 2),
            pretrained=config.get('pretrained', True),
            dropout=config.get('dropout', 0.3)
        )
    else:
        raise ValueError(f"Unsupported model type: {model_type}")
    
    return model

## 2. Advanced Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class LabelSmoothingLoss(nn.Module):
    """Label smoothing for regularization"""
    
    def __init__(self, num_classes, smoothing=0.1):
        super(LabelSmoothingLoss, self).__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
    
    def forward(self, inputs, targets):
        log_probs = F.log_softmax(inputs, dim=1)
        targets_one_hot = torch.zeros_like(log_probs).scatter(1, targets.unsqueeze(1), 1)
        
        # Apply label smoothing
        targets_smooth = (1 - self.smoothing) * targets_one_hot + \
                        self.smoothing / self.num_classes
        
        loss = -(targets_smooth * log_probs).sum(dim=1).mean()
        return loss

def get_loss_function(config):
    """Get loss function based on configuration"""
    loss_type = config.get('loss_type', 'cross_entropy')
    
    if loss_type == 'cross_entropy':
        return nn.CrossEntropyLoss()
    elif loss_type == 'focal':
        return FocalLoss(
            alpha=config.get('focal_alpha', 1),
            gamma=config.get('focal_gamma', 2)
        )
    elif loss_type == 'label_smoothing':
        return LabelSmoothingLoss(
            num_classes=config.get('num_classes', 2),
            smoothing=config.get('label_smoothing', 0.1)
        )
    else:
        raise ValueError(f"Unsupported loss type: {loss_type}")

## 3. Training Utilities

In [ ]:
class EarlyStopping:
    """Early stopping to prevent overfitting"""
    
    def __init__(self, patience=7, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.best_loss = None
        self.counter = 0
        self.best_weights = None
    
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.save_checkpoint(model)
        else:
            self.counter += 1
        
        if self.counter >= self.patience:
            if self.restore_best_weights:
                model.load_state_dict(self.best_weights)
            return True
        return False
    
    def save_checkpoint(self, model):
        self.best_weights = model.state_dict().copy()

def mixup_data(x, y, alpha=1.0):
    """Apply mixup augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Mixup loss calculation"""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

class Trainer:
    """Training manager for detection model"""
    
    def __init__(self, model, train_loader, val_loader, config):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        
        # Loss function
        self.criterion = get_loss_function(config)
        
        # Optimizer
        optimizer_type = config.get('optimizer', 'adamw')
        if optimizer_type == 'adamw':
            self.optimizer = optim.AdamW(
                model.parameters(),
                lr=config.get('learning_rate', 1e-4),
                weight_decay=config.get('weight_decay', 1e-4)
            )
        elif optimizer_type == 'sgd':
            self.optimizer = optim.SGD(
                model.parameters(),
                lr=config.get('learning_rate', 1e-3),
                momentum=config.get('momentum', 0.9),
                weight_decay=config.get('weight_decay', 1e-4)
            )
        
        # Scheduler
        scheduler_type = config.get('scheduler', 'reduce_lr')
        if scheduler_type == 'reduce_lr':
            self.scheduler = ReduceLROnPlateau(
                self.optimizer, mode='min', factor=0.5, patience=5, verbose=True
            )
        elif scheduler_type == 'cosine':
            self.scheduler = CosineAnnealingLR(
                self.optimizer, T_max=config.get('epochs', 100)
            )
        
        # Early stopping
        self.early_stopping = EarlyStopping(
            patience=config.get('patience', 10),
            min_delta=config.get('min_delta', 1e-4)
        )
        
        # Training history
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_acc': [],
            'val_acc': []
        }
    
    def train_epoch(self):
        """Train for one epoch"""
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        progress_bar = tqdm(self.train_loader, desc='Training')
        
        for batch_idx, (inputs, targets) in enumerate(progress_bar):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Apply mixup if enabled
            use_mixup = self.config.get('use_mixup', False)
            if use_mixup and np.random.random() > 0.5:
                inputs, targets_a, targets_b, lam = mixup_data(
                    inputs, targets, alpha=self.config.get('mixup_alpha', 1.0)
                )
                
                self.optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = mixup_criterion(self.criterion, outputs, targets_a, targets_b, lam)
            else:
                self.optimizer.zero_grad()
                outputs = self.model(inputs)
                loss = self.criterion(outputs, targets)
            
            loss.backward()
            
            # Gradient clipping
            if self.config.get('grad_clip', 0) > 0:
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), self.config['grad_clip']
                )
            
            self.optimizer.step()
            
            # Statistics
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            
            if not use_mixup or np.random.random() <= 0.5:
                correct += predicted.eq(targets).sum().item()
            
            # Update progress bar
            progress_bar.set_postfix({
                'Loss': f'{running_loss/(batch_idx+1):.4f}',
                'Acc': f'{100.*correct/total:.2f}%'
            })
        
        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = 100. * correct / total
        
        return epoch_loss, epoch_acc
    
    def validate_epoch(self):
        """Validate for one epoch"""
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            progress_bar = tqdm(self.val_loader, desc='Validation')
            
            for batch_idx, (inputs, targets) in enumerate(progress_bar):
                inputs, targets = inputs.to(device), targets.to(device)
                
                outputs = self.model(inputs)
                loss = self.criterion(outputs, targets)
                
                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
                
                progress_bar.set_postfix({
                    'Loss': f'{running_loss/(batch_idx+1):.4f}',
                    'Acc': f'{100.*correct/total:.2f}%'
                })
        
        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = 100. * correct / total
        
        return epoch_loss, epoch_acc
    
    def train(self, epochs):
        """Full training loop"""
        print(f"Starting training for {epochs} epochs...")
        
        for epoch in range(epochs):
            print(f"\nEpoch {epoch+1}/{epochs}")
            print("-" * 50)
            
            # Train
            train_loss, train_acc = self.train_epoch()
            
            # Validate
            val_loss, val_acc = self.validate_epoch()
            
            # Update history
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)
            self.history['train_acc'].append(train_acc)
            self.history['val_acc'].append(val_acc)
            
            # Scheduler step
            if isinstance(self.scheduler, ReduceLROnPlateau):
                self.scheduler.step(val_loss)
            else:
                self.scheduler.step()
            
            # Print results
            print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
            print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
            
            # Early stopping
            if self.early_stopping(val_loss, self.model):
                print(f"Early stopping triggered after epoch {epoch+1}")
                break
        
        print("Training completed!")
        return self.history

## 4. Model Configuration and Setup

In [ ]:
# Detection model configuration
DETECTION_CONFIG = {
    # Model architecture
    'model_type': 'efficientnet',
    'backbone': 'efficientnet-b4',
    'num_classes': 2,
    'pretrained': True,
    'dropout': 0.3,
    
    # Training parameters
    'epochs': 100,
    'batch_size': 16,
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'optimizer': 'adamw',
    'scheduler': 'reduce_lr',
    
    # Loss function
    'loss_type': 'focal',  # 'cross_entropy', 'focal', 'label_smoothing'
    'focal_alpha': 1,
    'focal_gamma': 2,
    'label_smoothing': 0.1,
    
    # Regularization
    'use_mixup': True,
    'mixup_alpha': 1.0,
    'grad_clip': 1.0,
    
    # Early stopping
    'patience': 10,
    'min_delta': 1e-4,
    
    # Data
    'data_dir': '../data',
    'num_workers': 4
}

# Save configuration
with open('detection_config.json', 'w') as f:
    json.dump(DETECTION_CONFIG, f, indent=2)

print("Detection model configuration:")
for key, value in DETECTION_CONFIG.items():
    print(f"  {key}: {value}")

## 5. Model Training (Demo with Dummy Data)

In [ ]:
# Create model
model = create_model(DETECTION_CONFIG)
print(f"Created {DETECTION_CONFIG['model_type']} model with {DETECTION_CONFIG['backbone']} backbone")

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Demo training with dummy data (replace with actual data loaders)
class DummyDataLoader:
    def __init__(self, batch_size=16, num_batches=10):
        self.batch_size = batch_size
        self.num_batches = num_batches
    
    def __len__(self):
        return self.num_batches
    
    def __iter__(self):
        for _ in range(self.num_batches):
            # Create dummy SAR images (3 channels for compatibility)
            images = torch.randn(self.batch_size, 3, 512, 512)
            labels = torch.randint(0, 2, (self.batch_size,))
            yield images, labels

# Create dummy data loaders for demonstration
print("\nCreating dummy data loaders for demonstration...")
train_loader = DummyDataLoader(batch_size=8, num_batches=5)
val_loader = DummyDataLoader(batch_size=8, num_batches=3)

# Test forward pass
print("\nTesting forward pass...")
dummy_input = torch.randn(1, 3, 512, 512).to(device)
model.eval()
with torch.no_grad():
    output = model(dummy_input)
    print(f"Input shape: {dummy_input.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Output: {output}")
    probs = F.softmax(output, dim=1)
    print(f"Probabilities: {probs}")

print("\nModel is ready for training!")
print("To train with real data:")
print("1. Replace dummy data loaders with actual SAR dataset")
print("2. Uncomment and run the training section below")

## 6. Training Loop (Uncomment when ready)

In [ ]:
# Uncomment this section when you have real data
"""
# Create data loaders with real data
data_loaders = create_data_loaders(
    DETECTION_CONFIG['data_dir'], 
    batch_size=DETECTION_CONFIG['batch_size'],
    num_workers=DETECTION_CONFIG['num_workers']
)

train_loader = data_loaders['detection']['train']
val_loader = data_loaders['detection']['val']

# Create trainer
trainer = Trainer(model, train_loader, val_loader, DETECTION_CONFIG)

# Start training
history = trainer.train(DETECTION_CONFIG['epochs'])

# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'config': DETECTION_CONFIG,
    'history': history
}, 'oil_spill_detector.pth')

print("Model saved as 'oil_spill_detector.pth'")
"""

print("Training section is commented out - uncomment when ready with real data")

## 7. Evaluation Functions

In [ ]:
def evaluate_model(model, test_loader, device):
    """Comprehensive model evaluation"""
    model.eval()
    
    all_predictions = []
    all_probabilities = []
    all_targets = []
    
    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc='Evaluating'):
            inputs = inputs.to(device)
            outputs = model(inputs)
            probabilities = F.softmax(outputs, dim=1)
            predictions = outputs.argmax(dim=1)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
            all_targets.extend(targets.numpy())
    
    all_predictions = np.array(all_predictions)
    all_probabilities = np.array(all_probabilities)
    all_targets = np.array(all_targets)
    
    return all_predictions, all_probabilities, all_targets

def plot_training_history(history):
    """Plot training curves"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss curves
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['val_loss'], label='Validation Loss', marker='s')
    axes[0].set_title('Model Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[1].plot(history['train_acc'], label='Train Accuracy', marker='o')
    axes[1].plot(history['val_acc'], label='Validation Accuracy', marker='s')
    axes[1].set_title('Model Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_evaluation_metrics(predictions, probabilities, targets):
    """Plot comprehensive evaluation metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Confusion Matrix
    cm = confusion_matrix(targets, predictions)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0])
    axes[0, 0].set_title('Confusion Matrix')
    axes[0, 0].set_ylabel('True Label')
    axes[0, 0].set_xlabel('Predicted Label')
    
    # ROC Curve
    if len(np.unique(targets)) == 2:  # Binary classification
        fpr, tpr, _ = roc_curve(targets, probabilities[:, 1])
        roc_auc = roc_auc_score(targets, probabilities[:, 1])
        
        axes[0, 1].plot(fpr, tpr, linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
        axes[0, 1].plot([0, 1], [0, 1], 'k--', linewidth=1)
        axes[0, 1].set_title('ROC Curve')
        axes[0, 1].set_xlabel('False Positive Rate')
        axes[0, 1].set_ylabel('True Positive Rate')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
    
        # Precision-Recall Curve
        precision, recall, _ = precision_recall_curve(targets, probabilities[:, 1])
        avg_precision = average_precision_score(targets, probabilities[:, 1])
        
        axes[1, 0].plot(recall, precision, linewidth=2, 
                       label=f'PR Curve (AP = {avg_precision:.3f})')
        axes[1, 0].set_title('Precision-Recall Curve')
        axes[1, 0].set_xlabel('Recall')
        axes[1, 0].set_ylabel('Precision')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
    
    # Classification Report
    report = classification_report(targets, predictions, output_dict=True)
    report_df = pd.DataFrame(report).iloc[:-1, :].T
    
    axes[1, 1].axis('tight')
    axes[1, 1].axis('off')
    table = axes[1, 1].table(cellText=report_df.round(3).values,
                            rowLabels=report_df.index,
                            colLabels=report_df.columns,
                            cellLoc='center',
                            loc='center')
    axes[1, 1].set_title('Classification Report')
    
    plt.tight_layout()
    plt.show()

print("Evaluation functions ready!")
print("\nNext steps:")
print("1. Prepare your SAR dataset")
print("2. Uncomment and run the training section")
print("3. Evaluate the trained model")
print("4. Proceed to segmentation model (03_Segmentation_Model.ipynb)")